In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
sentence_embedding = torch.tensor([
    [1.0, 0.0, 0.0],  # 我
    [0.0, 1.0, 0.0],  # 喜欢
    [0.0, 0.0, 1.0],  # 机器
    [0.5, 0.5, 0.0],  # 学习
])

print(sentence_embedding)
print(sentence_embedding.shape)

tensor([[1.0000, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 1.0000],
        [0.5000, 0.5000, 0.0000]])
torch.Size([4, 3])


In [3]:
scores = torch.matmul(sentence_embedding, sentence_embedding.T)

print(scores)

tensor([[1.0000, 0.0000, 0.0000, 0.5000],
        [0.0000, 1.0000, 0.0000, 0.5000],
        [0.0000, 0.0000, 1.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.5000]])


In [4]:
attention_weights = F.softmax(scores, dim=1)

print(attention_weights)

tensor([[0.4269, 0.1571, 0.1571, 0.2589],
        [0.1571, 0.4269, 0.1571, 0.2589],
        [0.1749, 0.1749, 0.4754, 0.1749],
        [0.2773, 0.2773, 0.1682, 0.2773]])


In [5]:
attention_output = torch.matmul(attention_weights, sentence_embedding)

print(attention_output)
print(attention_output.shape)

tensor([[0.5564, 0.2865, 0.1571],
        [0.2865, 0.5564, 0.1571],
        [0.2623, 0.2623, 0.4754],
        [0.4159, 0.4159, 0.1682]])
torch.Size([4, 3])


In [6]:
def simple_attention(x):
    scores = torch.matmul(x, x.T)
    weights = F.softmax(scores, dim=1)
    output = torch.matmul(weights, x)
    return output, weights

In [7]:
output, weights = simple_attention(sentence_embedding)

print("Attention 权重：")
print(weights)

print("Attention 输出：")
print(output)

Attention 权重：
tensor([[0.4269, 0.1571, 0.1571, 0.2589],
        [0.1571, 0.4269, 0.1571, 0.2589],
        [0.1749, 0.1749, 0.4754, 0.1749],
        [0.2773, 0.2773, 0.1682, 0.2773]])
Attention 输出：
tensor([[0.5564, 0.2865, 0.1571],
        [0.2865, 0.5564, 0.1571],
        [0.2623, 0.2623, 0.4754],
        [0.4159, 0.4159, 0.1682]])


In [8]:
embedding_dim = 3

W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_k = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_v = nn.Linear(embedding_dim, embedding_dim, bias=False)

Q = W_q(sentence_embedding)
K = W_k(sentence_embedding)
V = W_v(sentence_embedding)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([4, 3])
K shape: torch.Size([4, 3])
V shape: torch.Size([4, 3])


In [9]:
scores = torch.matmul(Q, K.T)

print(scores)
print(scores.shape)

tensor([[-0.0186, -0.0593, -0.0013, -0.0389],
        [-0.0674, -0.0316, -0.0362, -0.0495],
        [ 0.2775, -0.2130,  0.2579,  0.0322],
        [-0.0430, -0.0455, -0.0188, -0.0442]], grad_fn=<MmBackward0>)
torch.Size([4, 4])


In [10]:
import math

d_k = K.shape[-1]

scores = torch.matmul(Q, K.T) / math.sqrt(d_k)

print(scores)

tensor([[-0.0107, -0.0342, -0.0008, -0.0225],
        [-0.0389, -0.0182, -0.0209, -0.0286],
        [ 0.1602, -0.1230,  0.1489,  0.0186],
        [-0.0248, -0.0262, -0.0108, -0.0255]], grad_fn=<DivBackward0>)


In [11]:
attention_weights = F.softmax(scores, dim=1)

print(attention_weights)

tensor([[0.2516, 0.2457, 0.2541, 0.2486],
        [0.2469, 0.2521, 0.2514, 0.2495],
        [0.2770, 0.2087, 0.2739, 0.2404],
        [0.2493, 0.2489, 0.2528, 0.2491]], grad_fn=<SoftmaxBackward0>)


In [12]:
attention_output = torch.matmul(attention_weights, V)

print(attention_output)
print(attention_output.shape)

tensor([[ 0.0270, -0.1193,  0.2578],
        [ 0.0228, -0.1194,  0.2583],
        [ 0.0534, -0.1176,  0.2528],
        [ 0.0249, -0.1193,  0.2581]], grad_fn=<MmBackward0>)
torch.Size([4, 3])


In [13]:
class SimpleSelfAttention(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.W_q = nn.Linear(embedding_dim, embedding_dim)
        self.W_k = nn.Linear(embedding_dim, embedding_dim)
        self.W_v = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)

        d_k = K.shape[-1]

        scores = torch.matmul(Q, K.T) / math.sqrt(d_k)

        attention_weights = F.softmax(scores, dim=1)

        output = torch.matmul(attention_weights, V)

        return output, attention_weights

In [14]:
attention = SimpleSelfAttention(embedding_dim=3)

output, weights = attention(sentence_embedding)

print("输出：")
print(output)

print("注意力权重：")
print(weights)

输出：
tensor([[ 0.3530, -0.6766, -0.1406],
        [ 0.3465, -0.6797, -0.1386],
        [ 0.3416, -0.6740, -0.1346],
        [ 0.3498, -0.6781, -0.1396]], grad_fn=<MmBackward0>)
注意力权重：
tensor([[0.2511, 0.2261, 0.2846, 0.2382],
        [0.2348, 0.2388, 0.2896, 0.2368],
        [0.2301, 0.2266, 0.3149, 0.2284],
        [0.2429, 0.2324, 0.2872, 0.2376]], grad_fn=<SoftmaxBackward0>)
